# Reading the tables

In [15]:
# Run this in the Silver notebook
spark.sql("CREATE SCHEMA IF NOT EXISTS lh_adventureworks_silver.sales")
spark.sql("CREATE SCHEMA IF NOT EXISTS lh_adventureworks_silver.production")
spark.sql("CREATE SCHEMA IF NOT EXISTS lh_adventureworks_silver.hr")

StatementMeta(, a9d926c0-4dc7-40a5-b796-02db54ab2495, 17, Finished, Available, Finished, False)

DataFrame[]

In [16]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# Read Bronze tables
df_header = spark.read.table("lh_adventureworks_bronze.dbo.Sales_SalesOrderHeader")
df_detail = spark.read.table("lh_adventureworks_bronze.dbo.Sales_SalesOrderDetail")
df_customer = spark.read.table("lh_adventureworks_bronze.dbo.Sales_Customer")
df_person = spark.read.table("lh_adventureworks_bronze.dbo.Person_Person")
df_salesperson = spark.read.table("lh_adventureworks_bronze.dbo.Sales_SalesPerson")

print("Header rows:", df_header.count())
print("Detail rows:", df_detail.count())
print("Customer rows:", df_customer.count())
print("Person rows:", df_person.count())
print("SalesPerson rows:", df_salesperson.count())

StatementMeta(, a9d926c0-4dc7-40a5-b796-02db54ab2495, 18, Finished, Available, Finished, False)

Header rows: 31465
Detail rows: 121317
Customer rows: 19820
Person rows: 19972
SalesPerson rows: 17


In [17]:
from pyspark.sql import functions as F

# ── Step 1: Header + Detail ──────────────────────────────────────────
df_sales = df_header.alias("h").join(
    df_detail.alias("d"),
    on="SalesOrderID",
    how="inner"
)

# ── Step 2: Join Customer ─────────────────────────────────────────────
df_customer_clean = df_customer.select(
    F.col("CustomerID").alias("c_CustomerID"),
    F.col("PersonID"),
    F.col("StoreID"),
    F.col("TerritoryID").alias("c_TerritoryID")
)

df_sales = df_sales.join(
    df_customer_clean,
    df_sales["CustomerID"] == df_customer_clean["c_CustomerID"],
    how="left"
).drop("c_CustomerID")

# ── Step 3: Join Person (resolve customer name) ───────────────────────
df_person_clean = df_person.select(
    F.col("BusinessEntityID").alias("p_BusinessEntityID"),
    F.col("FirstName"),
    F.col("LastName")
)

df_sales = df_sales.join(
    df_person_clean,
    df_sales["PersonID"] == df_person_clean["p_BusinessEntityID"],
    how="left"
).drop("p_BusinessEntityID")

# ── Step 4: Join SalesPerson ──────────────────────────────────────────
df_sp_clean = df_salesperson.select(
    F.col("BusinessEntityID").alias("sp_BusinessEntityID"),
    F.col("SalesQuota"),
    F.col("Bonus")
)

df_sales = df_sales.join(
    df_sp_clean,
    df_sales["SalesPersonID"] == df_sp_clean["sp_BusinessEntityID"],
    how="left"
).drop("sp_BusinessEntityID")

# ── Step 5: Build full customer name ──────────────────────────────────
df_sales = df_sales.withColumn(
    "CustomerFullName",
    F.when(
        F.col("FirstName").isNotNull(),
        F.concat_ws(" ", F.col("FirstName"), F.col("LastName"))
    ).otherwise(F.lit("Store/Unknown Customer"))
)

# ── Step 6: Dedup + reject null PKs ──────────────────────────────────
df_sales = df_sales.dropDuplicates(["SalesOrderID", "SalesOrderDetailID"])
df_sales = df_sales.filter(
    F.col("SalesOrderID").isNotNull() & 
    F.col("SalesOrderDetailID").isNotNull()
)

# print("Final row count:", df_sales.count())
# df_sales.printSchema()

StatementMeta(, a9d926c0-4dc7-40a5-b796-02db54ab2495, 19, Finished, Available, Finished, False)

In [18]:
# ── Step 7: Clean up duplicate and redundant columns ─────────────────

# Drop duplicate rowguid and ModifiedDate from detail side
# Keep header versions (h.), rename for clarity
df_sales = df_sales \
    .drop("c_TerritoryID") \
    .withColumnRenamed("rowguid", "header_rowguid") \
    .withColumnRenamed("ModifiedDate", "header_ModifiedDate")

# The second rowguid and ModifiedDate are from SalesOrderDetail
# Rename those too — but since they're duplicates in the schema,
# we need to select explicitly to separate them
df_sales = df_sales.toDF(*[
    c if i not in [34, 35] else ["detail_rowguid", "detail_ModifiedDate"][i - 34]
    for i, c in enumerate(df_sales.columns)
])

# ── Step 8: Cast short/byte types to integer ─────────────────────────
df_sales = df_sales \
    .withColumn("OrderQty", F.col("OrderQty").cast("integer")) \
    .withColumn("Status", F.col("Status").cast("integer")) \
    .withColumn("RevisionNumber", F.col("RevisionNumber").cast("integer"))

# ── Step 9: Final column selection — only what Silver needs ──────────
df_silver_sales = df_sales.select(
    "SalesOrderID", "SalesOrderDetailID",
    "OrderDate", "DueDate", "ShipDate",
    "SalesOrderNumber", "Status", "OnlineOrderFlag",
    "CustomerID", "CustomerFullName", "PersonID", "StoreID",
    "SalesPersonID", "SalesQuota", "Bonus",
    "TerritoryID",
    "ProductID", "SpecialOfferID",
    "OrderQty", "UnitPrice", "UnitPriceDiscount", "LineTotal",
    "SubTotal", "TaxAmt", "Freight", "TotalDue",
    "PurchaseOrderNumber", "AccountNumber",
    "BillToAddressID", "ShipToAddressID",
    "CreditCardID", "CurrencyRateID",
    "header_rowguid", "header_ModifiedDate"
)

# print("Final column count:", len(df_silver_sales.columns))
# print("Final row count:", df_silver_sales.count())
# df_silver_sales.printSchema()

StatementMeta(, a9d926c0-4dc7-40a5-b796-02db54ab2495, 20, Finished, Available, Finished, False)

In [6]:
#display(df_sales.limit(100))

StatementMeta(, a9d926c0-4dc7-40a5-b796-02db54ab2495, 8, Finished, Available, Finished, False)

In [20]:
import logging
from pyspark.sql import functions as F

# ── Configure structured logger ───────────────────────────────────────
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("sales.sales_orders.dq")

dq_passed = True
dq_results = []

def log_check(name, passed, message):
    global dq_passed
    status = "PASS" if passed else "FAIL"
    log_msg = f"[DQ] [{status}] {name} — {message}"
    if passed:
        logger.info(log_msg)
    else:
        logger.error(log_msg)
        dq_passed = False
    dq_results.append({"check": name, "status": status, "message": message})

# ── 1. Row count guard ────────────────────────────────────────────────
expected_rows = df_detail.count()
actual_rows = df_silver_sales.count()
row_diff = actual_rows - expected_rows

log_check(
    "Row count",
    row_diff == 0,
    f"Expected {expected_rows:,}, got {actual_rows:,} (diff: {row_diff:+,})"
)

# ── 2. Null check on critical columns ────────────────────────────────
critical_cols = [
    "SalesOrderID", "SalesOrderDetailID", "CustomerID",
    "TerritoryID", "ProductID", "OrderDate", "LineTotal"
]
for col in critical_cols:
    null_count = df_silver_sales.filter(F.col(col).isNull()).count()
    log_check(
        f"Null check — {col}",
        null_count == 0,
        f"{null_count:,} nulls found"
    )

# ── 3. Duplicate PK check ─────────────────────────────────────────────
total = df_silver_sales.count()
distinct = df_silver_sales.select(
    "SalesOrderID", "SalesOrderDetailID"
).distinct().count()
log_check(
    "Duplicate PK",
    total == distinct,
    f"{total - distinct:,} duplicate PKs detected"
)

# ── 4. Join coverage ──────────────────────────────────────────────────
unmatched_customers = df_silver_sales.filter(
    F.col("CustomerFullName") == "Store/Unknown Customer"
).count()
unmatched_pct = (unmatched_customers / actual_rows) * 100
log_check(
    "Customer join coverage",
    unmatched_pct < 10,  # fail if more than 10% unmatched
    f"{unmatched_customers:,} unmatched ({unmatched_pct:.1f}%)"
)

# ── 5. Referential integrity ──────────────────────────────────────────
orphaned_customers = df_silver_sales.join(
    df_customer.select("CustomerID"),
    on="CustomerID",
    how="left_anti"
).count()
log_check(
    "Referential integrity — CustomerID",
    orphaned_customers == 0,
    f"{orphaned_customers:,} orphaned CustomerIDs"
)

# ── Hard fail if any critical check failed ────────────────────────────
failed_checks = [r for r in dq_results if r["status"] == "FAIL"]
if failed_checks:
    failed_names = ", ".join([r["check"] for r in failed_checks])
    raise Exception(
        f"silver.sales_orders DQ failed — {len(failed_checks)} check(s) failed: "
        f"{failed_names}. Pipeline halted to prevent bad data reaching Gold."
    )

logger.info(f"[DQ] All checks passed — {actual_rows:,} rows cleared for Silver write.")

StatementMeta(, a9d926c0-4dc7-40a5-b796-02db54ab2495, 22, Finished, Available, Finished, False)

INFO:sales.sales_orders.dq:[DQ] [PASS] Row count — Expected 121,317, got 121,317 (diff: +0)
INFO:sales.sales_orders.dq:[DQ] [PASS] Null check — SalesOrderID — 0 nulls found
INFO:sales.sales_orders.dq:[DQ] [PASS] Null check — SalesOrderDetailID — 0 nulls found
INFO:sales.sales_orders.dq:[DQ] [PASS] Null check — CustomerID — 0 nulls found
INFO:sales.sales_orders.dq:[DQ] [PASS] Null check — TerritoryID — 0 nulls found
INFO:sales.sales_orders.dq:[DQ] [PASS] Null check — ProductID — 0 nulls found
INFO:sales.sales_orders.dq:[DQ] [PASS] Null check — OrderDate — 0 nulls found
INFO:sales.sales_orders.dq:[DQ] [PASS] Null check — LineTotal — 0 nulls found
INFO:sales.sales_orders.dq:[DQ] [PASS] Duplicate PK — 0 duplicate PKs detected
INFO:sales.sales_orders.dq:[DQ] [PASS] Customer join coverage — 0 unmatched (0.0%)
INFO:sales.sales_orders.dq:[DQ] [PASS] Referential integrity — CustomerID — 0 orphaned CustomerIDs
INFO:sales.sales_orders.dq:[DQ] All checks passed — 121,317 rows cleared for Silver wr

In [21]:
# ── Write to silver Lakehouse ─────────────────────────────────────────
(
    df_silver_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("sales.sales_orders")
)
#print("dbo.sales_orders written successfully.")

StatementMeta(, a9d926c0-4dc7-40a5-b796-02db54ab2495, 23, Finished, Available, Finished, False)

In [12]:
# Quick verify
# df_verify = spark.read.table("dbo.sales_orders")
# print("Verified row count:", df_verify.count())

StatementMeta(, a9d926c0-4dc7-40a5-b796-02db54ab2495, 14, Finished, Available, Finished, False)

Verified row count: 121317
